# NDI-F benchmark on Google Colab

Runs the complete pipeline of the paper (*Normal Deviate Imputation (NDI-F): Closed-Form Factor-Model Imputation of
Numerical Tabular Data with a Quantified Accuracy–Cost Trade-off*) and rebuilds its tables and figures.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. The NDI family and the classical baselines need only the CPU;
GAIN, MIWAE, Sinkhorn imputation and ReMasker are much faster on the GPU. High-RAM is not needed.

**Time:** smoke test about 10 minutes; full run about 12.5 hours. The run is resumable: if the session ends, run the
cells again and it continues where it stopped. For the full run, set `USE_DRIVE = True` in the first cell so that the
results survive the end of a Colab session.

The repository already contains the results reported in the paper (`results/`); this notebook writes a new run to
`results_rerun/` and compares the two at the end.

In [ ]:
# 1) Working folder and repository
import os
USE_DRIVE = False          # True: keep everything in Google Drive (recommended for the full run)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive'
else:
    BASE = '/content'
os.chdir(BASE)
if not os.path.exists('ndi-imputation'):
    !git clone -q https://github.com/kurbankotan/ndi-imputation.git
%cd ndi-imputation

In [ ]:
# 2) Packages and the official ReMasker code (about 3-5 minutes, once)
!pip -q install "pandas<3" "scikit-learn>=1.5,<1.8" "hyperimpute==0.1.17" timm
if not os.path.exists('remasker/remasker_impute.py'):
    !git clone -q https://github.com/tydusky/remasker.git
# two compatibility fixes in the ReMasker code (an unused tkinter import; numpy aliases removed in numpy 2)
!sed -i 's/^from tkinter import E$/# from tkinter import E/' remasker/model_mae.py
!sed -i 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g' remasker/remasker_impute.py remasker/model_mae.py remasker/utils.py remasker/plugin_mae.py
import multiprocessing, torch
print("CPU cores:", multiprocessing.cpu_count(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# 3) Smoke test (about 10 minutes) -> results_quick/
!PYTHONHASHSEED=3 QUICK=1 python ndi_benchmark.py 2>&1 | grep -v -i "warning\|deprecat" | tail -n 40

In [ ]:
# 4) Full run as in the paper (about 12.5 hours) -> results_rerun/
#    DEEP_SEEDS=0: the four learning-based baselines use the first seed, as in the paper (use 0,1,2 for all three seeds)
#    PYTHONHASHSEED=3: the order in which HyperImpute tries its candidate learners, as in the paper's run (see README)
!PYTHONHASHSEED=3 DEEP_SEEDS=0 OUT_DIR=results_rerun python ndi_benchmark.py 2>&1 | grep -v -i "warning\|deprecat" | tee -a log_rerun.txt

In [ ]:
# 5) Tables (tables.json) and figures of the paper from the new run
%cd results_rerun
!python ../make_tables.py > /dev/null && python ../make_figures.py
%cd ..

In [ ]:
# 6) Compare with the results reported in the paper
import json
paper = json.load(open('results/tables.json'))
rerun = json.load(open('results_rerun/tables.json'))
for m in ['MissForest', 'GaussCM', 'SoftImpute', 'NDI-F', 'MICE', 'Mean', 'NDI-0']:
    print(f"{m:<12s} pooled rank  paper {paper['friedman']['pooled']['ranks'][m]:5.2f}   rerun {rerun['friedman']['pooled']['ranks'][m]:5.2f}")

In [ ]:
# 7) Download the new results
from google.colab import files
files.download('results_rerun.zip')